In [7]:
import pandas as pd
from pathlib import Path

base_dir = Path("/Users/soorykantsuman/Desktop/deidentification/NOTEBOOK/MIND")

phi_custom = pd.read_csv(base_dir / "phi_custom.csv")
phi_p0 = pd.read_csv(base_dir / "phi_pr1_p0.csv")
phi_p1 = pd.read_csv(base_dir / "phi_pr1_p1.csv")
phi_p2 = pd.read_csv(base_dir / "phi_pr1_p2.csv")

phi_pr_files = {
    "phi_pr1_p0": phi_p0,
    "phi_pr1_p1": phi_p1,
    "phi_pr1_p2": phi_p2,
}

tables_custom = sorted(phi_custom["TABLE_NAME"].unique())

table_locations = {}
for tbl in tables_custom:
    locations = []
    for name, df in phi_pr_files.items():
        if tbl in df["TABLE_NAME"].values:
            locations.append(name)
    table_locations[tbl] = locations

result = {}
# Display mapping: table -> which phi_pr1_* files it appears in
for tbl, locs in table_locations.items():
    loc = locs[0]
    if loc not in result:
        result[loc] = []
    result[loc].append(tbl)
    print(f"{tbl}: {', '.join(locs) if locs else 'NOT FOUND'}")
print(result)

Address: phi_pr1_p0
CareProvider: phi_pr1_p0
CareProviderLocation: phi_pr1_p0
CareProviderSpecialty: phi_pr1_p0
Countries: phi_pr1_p0
Ethnicity: phi_pr1_p0
Gender: phi_pr1_p0
Languages: phi_pr1_p0
MaritalStatus: phi_pr1_p0
Patient: phi_pr1_p0
Person: phi_pr1_p1
PlaceOfService: phi_pr1_p0
Race: phi_pr1_p0
ServiceGroup: phi_pr1_p1
Visit: phi_pr1_p0
br_Account: phi_pr1_p1
br_ServiceDetail: phi_pr1_p1
fctPatientInsurance: phi_pr1_p1
{'phi_pr1_p0': ['Address', 'CareProvider', 'CareProviderLocation', 'CareProviderSpecialty', 'Countries', 'Ethnicity', 'Gender', 'Languages', 'MaritalStatus', 'Patient', 'PlaceOfService', 'Race', 'Visit'], 'phi_pr1_p1': ['Person', 'ServiceGroup', 'br_Account', 'br_ServiceDetail', 'fctPatientInsurance']}


In [9]:
for key, value in result.items():
    print(key, len(value), value)

phi_pr1_p0 13 ['Address', 'CareProvider', 'CareProviderLocation', 'CareProviderSpecialty', 'Countries', 'Ethnicity', 'Gender', 'Languages', 'MaritalStatus', 'Patient', 'PlaceOfService', 'Race', 'Visit']
phi_pr1_p1 5 ['Person', 'ServiceGroup', 'br_Account', 'br_ServiceDetail', 'fctPatientInsurance']


In [2]:
cols_key = ["TABLE_NAME", "COLUMN_NAME"]
cols_cfg = ["IS_PHI", "DE_IDENTIFICATION_RULE", "MASK_VALUE",
            "REFERENCE_PATIENT_ID", "REFERENCE_ENCOUNTER_ID"]

def apply_custom_updates(target_df, custom_df):
    # Index on table+column
    target_idx = target_df.set_index(cols_key)
    custom_idx = custom_df.set_index(cols_key)[cols_cfg]

    common_index = target_idx.index.intersection(custom_idx.index)
    print(f"Updating {len(common_index)} (TABLE_NAME, COLUMN_NAME) pairs")

    target_idx.loc[common_index, cols_cfg] = custom_idx.loc[common_index, cols_cfg]
    return target_idx.reset_index()

updated = {}
for name, df in phi_pr_files.items():
    # Only custom rows for tables present in this phi_pr file
    tables_here = df["TABLE_NAME"].unique()
    custom_sub = phi_custom[phi_custom["TABLE_NAME"].isin(tables_here)]
    print(f"\n=== {name} ===")
    updated[name] = apply_custom_updates(df, custom_sub)

# Write updated versions (non-destructive: write to *_updated.csv)
for name, df_upd in updated.items():
    out_path = base_dir / f"{name}.csv"
    df_upd.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")


=== phi_pr1_p0 ===
Updating 183 (TABLE_NAME, COLUMN_NAME) pairs

=== phi_pr1_p1 ===
Updating 322 (TABLE_NAME, COLUMN_NAME) pairs

=== phi_pr1_p2 ===
Updating 27 (TABLE_NAME, COLUMN_NAME) pairs
Wrote /Users/soorykantsuman/Desktop/deidentification/NOTEBOOK/MIND/phi_pr1_p0.csv
Wrote /Users/soorykantsuman/Desktop/deidentification/NOTEBOOK/MIND/phi_pr1_p1.csv
Wrote /Users/soorykantsuman/Desktop/deidentification/NOTEBOOK/MIND/phi_pr1_p2.csv
